In [ ]:
!pip install gradio pymupdf scikit-learn gdown requests --quiet
# ==========================================
# gradio            : UI liberary that allows to turn the code into a HTML like page (same structure that we will use with html that allows css and html combination with table)
# pymupdf           : liberary known as fitz allows to open PDF files and extract text from them while keeping the structure and location(page) of each sentence
# scikit-learn      : machine learning Liberary - used for TF-IDF aswell as Cosine Similarity
# TF-IDF            : builds dictionary(creates vectoric representaion that allows Cosine Similarity between questions to data sources) from articles and allows to search words -transfers Data From articles to the rest of the modele- the Retriever of the RAG
# Cosine Similarity : allows for mathematical calculations to detirmine how close the our question is to the given Texts(articles) - second part of the Retriever of the RAG
# gdown             : Liberary that we use to bypass Google Drives Defences in order to download files from it for IDES like Colab(we keep the articles on our google Drive to be used at the begining of each run)
# requests          : Gives the Code API to communicate and make http requests to Gemini servers. -- the Generator of RAG
# --quite           : cosmetic - Dont Show the texts that will be printed from intalling the Libereris
# ==========================================

import os
import re
import requests
import gradio as gr
import fitz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import plotly.express as px
import datetime
import random
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import imageio
gr.close_all()# closeing all Previosly opened interfaces to clear memory -- prevents Port colitions

# ==========================================
#  GEMINI has a few possible version and sometimes some of them are unavailable which makes the program crush
#  so we try each on and if its unavailable we downgrade to the next possibility (Code Robustness)
#  1. handshake with Googles Gemini
#  2. sends the key to recive allowed Ai modeles
#  3. Filters to the modeles that support generateContent(the Generator)
#  4. prioritises FlashModels after that pro (stronger) models and uses the best possible option
# ==========================================
API_KEY = ""
ACTIVE_MODEL = "models/gemini-1.5-flash" #default value will change in discovery_model if its unavailable

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 26.8 MB/s eta 0:00:00


In [2]:
def discovery_model():#Fallback Mechanism
#{
    global ACTIVE_MODEL
    try:
        url = f"https://generativelanguage.googleapis.com/v1beta/models?key={API_KEY}"
        res = requests.get(url, timeout=10)#get the available models for my api key , 10 sec wait time to avoid deadlocks

        if res.status_code == 200:#200 == success
            models = res.json().get('models', [])# get the answers in a json dictionary type
            valid_models = [m['name'] for m in models if 'generateContent' in m.get('supportedGenerationMethods', [])] #check if the recived model can work with generateContent (can answer questions)

            if valid_models:#priritizing models
                flash = [m for m in valid_models if 'flash' in m] # does the model contain the word flash? if so add the model to flash list
                pro = [m for m in valid_models if 'pro' in m]#does the model contain the word pro?if so add the model to pro list

                if flash: ACTIVE_MODEL = flash[0]#prioritize flash
                elif pro: ACTIVE_MODEL = pro[0]
                else: ACTIVE_MODEL = valid_models[0]

                print(f"Active Model: {ACTIVE_MODEL}")
    except Exception:
        pass
#}
#discovery_model()

In [3]:
def call_gemini_direct(prompt):
  # ==========================================
  # This is the generator:
  # send the datato the Ai model
  # ==========================================
  #{
    url = f"https://generativelanguage.googleapis.com/v1beta/{ACTIVE_MODEL}:generateContent?key={API_KEY}"#here we ask for a specific model instead of a list like before(Model Invocation)
    headers = {'Content-Type': 'application/json'}#the is the type of format the data will be sent to the AI API
    data = {"contents": [{"parts": [{"text": prompt}]}]}#the needed format to be sent to the AI Generator - json Frame

    response = requests.post(url, headers=headers, json=data, timeout=15)# send the data to the Ai Model wait for up to 15 sec
    response.raise_for_status() #catch errors
    return response.json()['candidates'][0]['content']['parts'][0]['text']#this is the data we get back from the Ai model, notice we do not need to use asyncronic cmds here
#}
#call_gemini_direct(prompt)

In [4]:
#Global variables for use to save data from articles
rag_chunks = []
vectorizer = None
tfidf_matrix = None

In [5]:
def clean_text(text):
  #{
    text = text.replace("-", " ").replace("\n", " ")
    return re.sub(r'\s+', ' ', text).strip()
  #}
  #cleat_text

In [6]:
def chunk_text(text, chunk_size=200, overlap=50):
  #{
    words = text.split() # split text into words
    chunks = []

    # step determines how much we jump forward.
    # chunk_size - overlap means we go back 50 words each time to avoid cutting sentences!
    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i+chunk_size])
        if len(chunk.strip()) > 50: # if we have less than 50 Characters left just enter all of then into the current chunk - avoid short chunks
            chunks.append(chunk)

    return chunks
  #}
#chunk_text

In [7]:
# ==========================================
# this part turns the pdf files into database
# ==========================================
def initialize_rag_system():
  #{
    global rag_chunks, vectorizer, tfidf_matrix
    rag_chunks = []
    base_download_path = "./papers"
    os.makedirs(base_download_path, exist_ok=True)#location where data will be saved, checks if path exist
    folder_url = "https://drive.google.com/drive/folders/1W_9CDVLOetS7hD-tjC7DoiEB-VvmrPgw"#location of pdfs in drive

    os.system(f"gdown --folder {folder_url} -O {base_download_path} --remaining-ok --quiet")# This call downloads the entire Google Drive folder to our local machine,
                                                                                            # ensuring all necessary articles are available for the indexing pipeline.

    final_folder = "./papers/papers" if (os.path.exists("./papers/papers") and any(f.endswith('.pdf') for f in os.listdir("./papers/papers"))) else "./papers"
    #incase path location i incorrect will allow to search in other locations for paper foldair
    pdf_files = [f for f in os.listdir(final_folder) if f.endswith(".pdf")]#get all pdf files into list

    for file_name in pdf_files:
        try:
            doc = fitz.open(os.path.join(final_folder, file_name))#open each pdf file into RAM
            for page_num, page in enumerate(doc):#enumirate allows us to get both page num and content
                raw_text = page.get_text("text", sort=True)
                if raw_text:
                    cleaned_text = clean_text(raw_text)#remove unwated chars
                    for chunk in chunk_text(cleaned_text):
                        rag_chunks.append({"file_name": file_name, "page": page_num + 1, "text": chunk}) # get chunks to be in Dictionary type that we can send to generator AI
        except Exception:
            pass#takes care of falude files

    if not rag_chunks:
        return "couldnt find chunks."

    documents = [chunk["text"] for chunk in rag_chunks]#list of all the chunks we found for later use

    vectorizer = TfidfVectorizer(stop_words="english")#ignore stop words from the english directory #noise filter # Initialize TF-IDF Vectorizer to convert text to numerical vectors
    tfidf_matrix = vectorizer.fit_transform(documents)# build directory from the document

    return f"RAG Ready! got  {len(rag_chunks) } chunks."
    #}
#initialize_rag_system()

In [8]:
#
# RAG SEARCH
#
def gradio_rag_interface(user_text):
  #{
    try:
        if not user_text.strip():#if empty text
            return "Please enter a question."

        clean_query = clean_text(user_text)#get reads of unwated characters


        synonym_prompt = f"Generate 5 scientific synonyms or related keywords for this query: '{clean_query}'. Return ONLY the words separated by spaces, nothing else."
        expanded_words = call_gemini_direct(synonym_prompt)
        super_query = clean_query + " " + expanded_words
        query_vector = vectorizer.transform([super_query])#turn the question into vector we can work with for TF-IDF

        similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()#compare the question vector to all the vectors in the answer matrix to find similar answer

        top_indices = similarities.argsort()[::-1]#reverse the answers so that they will be ranked from best to worst #argsort sorts the answers as to be used in rankj
        relevant_chunks = []#saves the asked questions
        seen_files = set()#saves the filenames that we took answers too from allready so that later will allow to draw answer for diverce sorces

        for idx in top_indices:#go throu the chunks from most similar to lowers if we didnt take info from this source add it to the list of relevent_chunks and add it to seen fils so we wont look at it again
            file_name = rag_chunks[idx]['file_name']
            if file_name not in seen_files:
                relevant_chunks.append(rag_chunks[idx])
                seen_files.add(file_name)

            if len(relevant_chunks) >= 4:#stop looking after 4 sources
                break

        context_text = ""
        #From here we have the interface!!
        html = "<div style='padding-right: 15px;'><h2 style='color:#2563eb;'> Retrieved Sources</h2>"

        for i, chunk in enumerate(relevant_chunks):#just adds the number of article to the file print for clarafication #only print first 350 chars since it might be to big for the interface
            html += f"""
            <div style="padding:15px; margin-bottom:10px; border-radius:10px; border-left:5px solid #2563eb; background: var(--background-fill-secondary); color: var(--body-text-color);">
            <b style='color:#2563eb;'>Source {i+1}</b><br>📄 File: {chunk['file_name']}<br>📑 Page: {chunk['page']}<br><br>
            <span style='font-size:14.5px;'>{chunk['text'][:350]}...</span>
            </div>
            """
            context_text += f"\nDOCUMENT: {chunk['file_name']}\nPAGE: {chunk['page']}\nCONTENT:\n{chunk['text']}\n"#this is what we send to gemini

        html += "</div>"

        prompt = f"""
        Answer the question ONLY using the provided context.
        Cite filenames and page numbers.
        Do NOT use any outside knowledge under any circumstances.
        If the answer is not fully found in the context, say exactly: "The answer is not found in the provided context."

        CONTEXT:
        {context_text}

        QUESTION:
        {user_text}
        """

        answer = call_gemini_direct(prompt)

        html += f"""
        <hr><div style="padding:20px; border-radius:10px; border:1px solid var(--border-color-primary); background: var(--background-fill-secondary); color: var(--body-text-color); line-height:1.6;">
        <h2 style='color:#2563eb; margin-top:0;'> RAG Answer</h2>
        <span style='font-size:15px; white-space: pre-wrap;'>{answer}</span>
        </div>
        """

        return html

    except Exception as e:
        return f"<div style='color:#b91c1c; padding:20px;'>Error: {str(e)}</div>"

            #}
            #def gradio_rag_interface(user_text):

In [9]:

# # UI

# custom_css = """
# .gradio-container { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; }
# #search-btn { background: linear-gradient(to right, #059669, #10b981) !important; border: none !important; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1); transition: all 0.2s ease-in-out; color: white !important;}
# #search-btn:hover { transform: translateY(-2px); box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1); }
# """

# with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="slate"), css=custom_css, title="CropSurvive Academic Engine") as demo:

#     gr.HTML("""
#     <div style="padding: 35px 20px; background: linear-gradient(135deg, #064e3b 0%, #059669 100%); border-radius: 16px; color: white; text-align: center; box-shadow: 0 10px 25px -5px rgba(5, 150, 105, 0.4); margin-bottom: 30px; position: relative; overflow: hidden;">
#         <div style="position: absolute; top: -15px; left: -15px; font-size: 120px; opacity: 0.05;">🌿</div>
#         <h1 style='margin:0; font-size: 2.8em; font-weight: 800; letter-spacing: -0.5px; text-shadow: 0 2px 4px rgba(0,0,0,0.2);'> CropSurvive</h1>
#         <p style='font-size: 1.2em; opacity: 0.9; margin-top: 8px; font-weight: 300;'>Academic Research Engine</p>
#         <div style="margin-top: 15px; display: inline-block; background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px; font-size: 0.9em; backdrop-filter: blur(5px); font-weight: 500;">
#             TF-IDF Indexing &nbsp;&nbsp; Strict RAG Generation
#         </div>
#     </div>
#     """)

#     with gr.Row():
#         with gr.Column(scale=4):

#             input_box = gr.Textbox(
#                 label="Enter your research query",
#                 placeholder="e.g., What are the main components of the biological coating?",
#                 lines=3,
#                 elem_id="search-box",
#                 show_label=True
#             )

#         with gr.Column(scale=1, min_width=120):

#             submit_btn = gr.Button("Search ", variant="primary", elem_id="search-btn")

#             clear_btn = gr.Button("Clear", variant="secondary")



#     gr.HTML("<hr style='margin-top: 20px; margin-bottom: 20px; border-top: 1px solid #e2e8f0;'/>")


#     output_html = gr.HTML()


#     submit_btn.click(fn=gradio_rag_interface, inputs=input_box, outputs=output_html)
#     input_box.submit(fn=gradio_rag_interface, inputs=input_box, outputs=output_html)

#     # Safe clear functionality
#     clear_btn.click(fn=lambda: ("", ""), inputs=None, outputs=[input_box, output_html])


# print(initialize_rag_system())
# discovery_model()
# demo.launch(share=True)



example of possible questions:
1.   What are the main components or active ingredients used to create the biological coating?
2.   What are the main functions of alginate-based edible coatings in fresh-cut produce, and how do they affect the product's shelf life?


In [10]:
def Download_XL_Data_From_Drive():
    base_download_path = "./Data"
    os.makedirs(base_download_path, exist_ok=True)
    folder_url = "https://drive.google.com/drive/folders/1W_9CDVLOetS7hD-tjC7DoiEB-VvmrPgw"
    os.system(f"gdown --folder {folder_url} -O {base_download_path} --remaining-ok --quiet")
#Download_XL_Data_From_Drive():

In [11]:
Download_XL_Data_From_Drive()

In [12]:
class xl_morphology:
  def __init__(self, day, treatment,morphology_rate):
    self.day = day
    self.treatment = treatment
    self.morphology_rate = [morphology_rate]
  def __repr__(self):
        return f"[Day: {self.day} | {self.treatment} | Rates: {self.morphology_rate}]"
#class xl_morphology:
class xl_bacteria_yeasts:
  def __init__(self, day, treatment,bacteria_total,yeasts):
    self.day = day
    self.treatment = treatment
    self.bacteria_total = [bacteria_total]
    self.yeasts = [yeasts]
  def __repr__(self):
        return f"[Day: {self.day} | {self.treatment} | Bacteria Total: {self.bacteria_total} | Yeasts: {self.yeasts}]"
#class xl_bacteria_yeasts:
class xl_bacteria_lactic:
  def __init__(self, day, treatment,bacteria_total,lactic):
    self.day = day
    self.treatment = treatment
    self.bacteria_total = [bacteria_total]
    self.lactic = [lactic]
  def __repr__(self):
        return f"[Day: {self.day} | {self.treatment} | Bacteria Total: {self.bacteria_total} | Lactic: {self.lactic}]"
#class xl_bacteria_lactic:
class xl_weight:
  def __init__(self, day, treatment,weight_loss):
    self.day = day
    self.treatment = treatment
    self.weight_loss = [weight_loss]
  def __repr__(self):
        return f"[Day: {self.day} | {self.treatment} | Weight Loss: {self.weight_loss}]"
#class xl_weight:
class xl_texture_analyzer:
  def __init__(self, day,treatment, force):
    self.day = day
    self.treatment = treatment
    self.force = [force]
  def __repr__(self):
        return f"[Day: {self.day} | {self.treatment} | Force: {self.force}]"
#class xl_texture_analyzer:
class xl_tss:
  def __init__(self, day,treatment, TSS):
    self.day = day
    self.treatment = treatment
    self.TSS = [TSS]
  def __repr__(self):
        return f"[Day: {self.day} | {self.treatment} | TSS: {self.TSS}]"
#class xl_tss:
class xl_o2:
  def __init__(self, day,treatment, o2):
    self.day = day
    self.treatment = treatment
    self.o2 = [o2]
  def __repr__(self):
        return f"[Day: {self.day} | {self.treatment} | O2: {self.o2}]"
#class xl_o2:
class xl_co2:
  def __init__(self, day,treatment, co2):
    self.day = day
    self.treatment = treatment
    self.co2 = [co2]
  def __repr__(self):
        return f"[Day: {self.day} | {self.treatment} | CO2: {self.co2}]"
#class xl_co2:
class xl_color:
  def __init__(self, day, treatment, delta_e):
    self.day = day
    self.treatment = treatment
    self.color = [delta_e]
  def __repr__(self):
        return f"[Day: {self.day} | {self.treatment} | Color (ΔE): {self.color}]"
#class xl_color:



In [13]:

# this part turns the XL files into database

def collect_data_from_xl_files_morphology():
  #{

   final_folder = "./Data/Data"
   excel_files = [f for f in os.listdir(final_folder) if f.endswith(".xlsx")]

   if not excel_files: return {}

   file_path = os.path.join(final_folder, excel_files[0])
   all_sheets = pd.read_excel(file_path, sheet_name="morphology", header=2)

   this_dictionary = {}

   if 'morphology_rate' in all_sheets:
       for index, row in all_sheets.iterrows():
            day = row['Time (day)']
            treatment = row['Treatment']
            rate = row['morphology_rate']
            key = (day, treatment)
            if key in this_dictionary:
                this_dictionary[key].morphology_rate.append(rate)
            else:
                this_dictionary[key] = xl_morphology(day, treatment, rate)
   return this_dictionary
    #}
#collect_data_from_xl_files_morphology()
def collect_data_from_xl_files_bacteria_total_yeasts():
  #{

   final_folder = "./Data/Data"
   excel_files = [f for f in os.listdir(final_folder) if f.endswith(".xlsx")]

   if not excel_files: return {}

   file_path = os.path.join(final_folder, excel_files[0])
   all_sheets = pd.read_excel(file_path, sheet_name="bacteria", header=2)

   this_dictionary = {}

   if 'bacteria_total' in all_sheets:
       for index, row in all_sheets.iterrows():
            day = row['Time (day)']
            treatment = row['Treatment']
            bacteria_val = row['bacteria_total']
            yeasts_val = row['yeasts']
            key = (day, treatment)
            if key in this_dictionary:
                this_dictionary[key].bacteria_total.append(bacteria_val)
                this_dictionary[key].yeasts.append(yeasts_val)
            else:
                this_dictionary[key] = xl_bacteria_yeasts(day, treatment, bacteria_val, yeasts_val)
   return this_dictionary
    #}
#collect_data_from_xl_files_bacteria_total_yeasts():
def collect_data_From_Xl_files_bacteria_total_lactic():
  #{

   final_folder = "./Data/Data"
   excel_files = [f for f in os.listdir(final_folder) if f.endswith(".xlsx")]

   if not excel_files: return {}

   file_path = os.path.join(final_folder, excel_files[0])
   all_sheets = pd.read_excel(file_path, sheet_name="bacteria", header=2)

   this_dictionary = {}

   if 'bacteria_total' in all_sheets:
       for index, row in all_sheets.iterrows():
            day = row['Time (day)']
            treatment = row['Treatment']
            bacteria_val = row['bacteria_total']
            lactic_val = row['lactic']
            key = (day, treatment)
            if key in this_dictionary:
                this_dictionary[key].bacteria_total.append(bacteria_val)
                this_dictionary[key].lactic.append(lactic_val)
            else:
                this_dictionary[key] = xl_bacteria_lactic(day, treatment, bacteria_val, lactic_val)
   return this_dictionary
    #}
#collect_data_From_Xl_files_bacteria_total_lactic():

def collect_data_from_xl_files_weight():
    #{
    final_folder = "./Data/Data"
    excel_files = [f for f in os.listdir(final_folder) if f.endswith(".xlsx")]

    if not excel_files: return {}

    file_path = os.path.join(final_folder, excel_files[0])
    all_sheets = pd.read_excel(file_path, sheet_name="weight", header=2)

    this_dictionary = {}

    if 'weight loss (%)' in all_sheets:
        for index, row in all_sheets.iterrows():
            day = row['Time (day)']
            treatment = row['Treatment']
            val = row['weight loss (%)']
            key = (day, treatment)

            if key in this_dictionary:
                this_dictionary[key].weight_loss.append(val)
            else:
                this_dictionary[key] = xl_weight(day, treatment, val)

    return this_dictionary
    #}
    #collect_data_from_xl_files_weight():

def collect_data_from_xl_files_texture_analyzer():
    #{
    final_folder = "./Data/Data"
    excel_files = [f for f in os.listdir(final_folder) if f.endswith(".xlsx")]

    if not excel_files: return {}

    file_path = os.path.join(final_folder, excel_files[0])
    all_sheets = pd.read_excel(file_path, sheet_name="Texture analyzer", header=2)

    this_dictionary = {}

    if 'Force (%)' in all_sheets:
        for index, row in all_sheets.iterrows():
            day = row['Time (day)']
            treatment = row['Treatment']
            val = row['Force (%)']
            key = (day, treatment)

            if key in this_dictionary:
                this_dictionary[key].force.append(val)
            else:
                this_dictionary[key] = xl_texture_analyzer(day, treatment, val)

    return this_dictionary
    #}
    #collect_data_from_xl_files_texture_analyzer()

def collect_data_from_xl_files_tss():
    #{
    final_folder = "./Data/Data"
    excel_files = [f for f in os.listdir(final_folder) if f.endswith(".xlsx")]

    if not excel_files: return {}

    file_path = os.path.join(final_folder, excel_files[0])
    all_sheets = pd.read_excel(file_path, sheet_name="TSS", header=2)

    this_dictionary = {}

    if 'TSS (%)' in all_sheets:
        for index, row in all_sheets.iterrows():
            day = row['Time (day)']
            treatment = row['Treatment']
            val = row['TSS (%)']
            key = (day, treatment)

            if key in this_dictionary:
                this_dictionary[key].TSS.append(val)
            else:
                this_dictionary[key] = xl_tss(day, treatment, val)

    return this_dictionary
    #}
    #collect_data_from_xl_files_tss()

def collect_data_from_xl_files_o2():
    #{
    final_folder = "./Data/Data"
    excel_files = [f for f in os.listdir(final_folder) if f.endswith(".xlsx")]

    if not excel_files: return {}

    file_path = os.path.join(final_folder, excel_files[0])
    all_sheets = pd.read_excel(file_path, sheet_name="O2", header=2)

    this_dictionary = {}

    if '%O2' in all_sheets:
        for index, row in all_sheets.iterrows():
            day = row['Time (day)']
            treatment = row['Treatment']
            val = row['%O2']
            key = (day, treatment)

            if key in this_dictionary:
                this_dictionary[key].o2.append(val)
            else:
                this_dictionary[key] = xl_o2(day, treatment, val)

    return this_dictionary
    #}
    #collect_data_from_xl_files_o2():

def collect_data_from_xl_files_co2():
    #{
    final_folder = "./Data/Data"
    excel_files = [f for f in os.listdir(final_folder) if f.endswith(".xlsx")]

    if not excel_files: return {}

    file_path = os.path.join(final_folder, excel_files[0])
    all_sheets = pd.read_excel(file_path, sheet_name="CO2", header=2)

    this_dictionary = {}

    if 'co2' in all_sheets:
        for index, row in all_sheets.iterrows():
            day = row['Time (day)']
            treatment = row['Treatment']
            val = row['co2']
            key = (day, treatment)

            if key in this_dictionary:
                this_dictionary[key].co2.append(val)
            else:
                this_dictionary[key] = xl_co2(day, treatment, val)

    return this_dictionary
    #}
    #collect_data_from_xl_files_co2():
def collect_data_from_xl_files_color():
    #{
    final_folder = "./Data/Data"
    excel_files = [f for f in os.listdir(final_folder) if f.endswith(".xlsx")]

    if not excel_files: return {}

    file_path = os.path.join(final_folder, excel_files[0])

    all_sheets = pd.read_excel(file_path, sheet_name="color", header=2)

    this_dictionary = {}

    if 'delta_E' in all_sheets:
        for index, row in all_sheets.iterrows():
            day = row['Time (day)']
            treatment = row['Treatment']
            val = row['delta_E']
            key = (day, treatment)

            if key in this_dictionary:
                this_dictionary[key].color.append(val)
            else:
                this_dictionary[key] = xl_color(day, treatment, val)

    return this_dictionary
    #}
#collect_data_from_xl_files_color()

In [14]:
#prints for debugging
print(collect_data_from_xl_files_morphology())
print(collect_data_from_xl_files_bacteria_total_yeasts())
print(collect_data_From_Xl_files_bacteria_total_lactic())
print(collect_data_from_xl_files_weight())
print(collect_data_from_xl_files_texture_analyzer())
print(collect_data_from_xl_files_tss())
print(collect_data_from_xl_files_o2())
print(collect_data_from_xl_files_co2())
print(collect_data_from_xl_files_color())

{(0, 'control'): [Day: 0 | control | Rates: [5, 5, 5]], (0, 'CMC-3-methyl'): [Day: 0 | CMC-3-methyl | Rates: [5, 5, 5]], (0, 'CMC-3-methyl+A'): [Day: 0 | CMC-3-methyl+A | Rates: [5, 5, 5]], (0, 'CMC-3-methyl+B'): [Day: 0 | CMC-3-methyl+B | Rates: [5, 5]], (0, 'CMC-3-methyl+C'): [Day: 0 | CMC-3-methyl+C | Rates: [5, 5, 5]], (4, 'control'): [Day: 4 | control | Rates: [5, 5]], (4, 'CMC-3-methyl'): [Day: 4 | CMC-3-methyl | Rates: [5, 5, 5]], (4, 'CMC-3-methyl+A'): [Day: 4 | CMC-3-methyl+A | Rates: [5, 5, 5]], (4, 'CMC-3-methyl+B'): [Day: 4 | CMC-3-methyl+B | Rates: [5, 5]], (4, 'CMC-3-methyl+C'): [Day: 4 | CMC-3-methyl+C | Rates: [5, 5]], (7, 'control'): [Day: 7 | control | Rates: [3, 5]], (7, 'CMC-3-methyl'): [Day: 7 | CMC-3-methyl | Rates: [4, 3, 3]], (7, 'CMC-3-methyl+A'): [Day: 7 | CMC-3-methyl+A | Rates: [3, 3, 3]], (7, 'CMC-3-methyl+B'): [Day: 7 | CMC-3-methyl+B | Rates: [5, 2, 4]], (7, 'CMC-3-methyl+C'): [Day: 7 | CMC-3-methyl+C | Rates: [3, 4, 3]]}
{(0, 'control'): [Day: 0 | contro

In [15]:
#prediction methods
def predict_Trendline_one_value_for_key(current_hash_data,time_to_predict,skip_value):
#{
    new_hash_Data = current_hash_data.copy()
    all_bacteria_types = set([key[1] for key in new_hash_Data.keys()])

    for bacteria_type in all_bacteria_types:
      historical_days = []
      historical_values = []
      for (day, b_type), obj in current_hash_data.items():
        if b_type == bacteria_type:
            values_list = list(vars(obj).values())[2]
            valid_values = [v for v in values_list if pd.notna(v)]
            if valid_values:
                    historical_days.append(day)
                    historical_values.append(sum(valid_values) / len(valid_values))
        #now we have all the days and their memoza
        n = len(set(historical_days))
        if n == 0:
            continue #no data


        if n >= 2:
            #only 2 days = linear more that 2 days exp = 2
            degree = 2 if n >= 3 else 1
            coefficients = np.polyfit(historical_days, historical_values, degree)
        for i in range(1, time_to_predict + 1):
            daytime = i * skip_value
            if daytime <= MAX_REAL_DAY:
                continue
            if n >= 2:
                predicted_value = np.polyval(coefficients, daytime)
            else:
                predicted_value = historical_values[0]

            new_hash_Data[(daytime, bacteria_type)] = predicted_value

    return new_hash_Data
#}
#def predict_Trendline_one_value_for_key(current_hash_data,time_to_predict,skip_value):
def predict_Trendline_two_values_per_key(current_hash_data,time_to_predict,skip_value):
    #{ same thing as befor but not we do it for two values per key
    new_hash_Data = current_hash_data.copy()
    all_bacteria_types = set([key[1] for key in new_hash_Data.keys()])

    for bacteria_type in all_bacteria_types:
        historical_days_1 = []
        historical_values_1 = []
        historical_days_2 = []
        historical_values_2 = []

        for (day, b_type), obj in current_hash_data.items():
            if b_type == bacteria_type:
                attributes = list(vars(obj).values())

                values_list_1 = attributes[2]
                values_list_2 = attributes[3]

                valid_val_1 = [v for v in values_list_1 if pd.notna(v)]
                if valid_val_1:
                    historical_days_1.append(day)
                    historical_values_1.append(sum(valid_val_1) / len(valid_val_1))

                valid_val_2 = [v for v in values_list_2 if pd.notna(v)]
                if valid_val_2:
                    historical_days_2.append(day)
                    historical_values_2.append(sum(valid_val_2) / len(valid_val_2))

        n1 = len(set(historical_days_1))
        n2 = len(set(historical_days_2))

        if n1 >= 2:
            degree_1 = 2 if n1 >= 3 else 1
            coeff_1 = np.polyfit(historical_days_1, historical_values_1, degree_1)

        if n2 >= 2:
            degree_2 = 2 if n2 >= 3 else 1
            coeff_2 = np.polyfit(historical_days_2, historical_values_2, degree_2)

        for i in range(1, time_to_predict + 1):
            daytime = i * skip_value
            if daytime <= MAX_REAL_DAY:
                continue
            if n1 >= 2:
                pred_1 = np.polyval(coeff_1, daytime)
            elif n1 == 1:
                pred_1 = historical_values_1[0]
            else:
                pred_1 = 0

            if n2 >= 2:
                pred_2 = np.polyval(coeff_2, daytime)
            elif n2 == 1:
                pred_2 = historical_values_2[0]
            else:
                pred_2 = 0
            new_hash_Data[(daytime, bacteria_type)] = (pred_1, pred_2)

    return new_hash_Data
    #}
#def predict_Trendline_two_values_per_key(current_hash_data,time_to_predict,skip_value):


In [16]:


#get XL data from database

dict_weight = collect_data_from_xl_files_weight()
dict_texture = collect_data_from_xl_files_texture_analyzer()
dict_tss = collect_data_from_xl_files_tss()
dict_o2 = collect_data_from_xl_files_o2()
dict_co2 = collect_data_from_xl_files_co2()
dict_morphology = collect_data_from_xl_files_morphology()
dict_bac_yeast = collect_data_from_xl_files_bacteria_total_yeasts()
dict_bac_lactic = collect_data_From_Xl_files_bacteria_total_lactic()
dict_color = collect_data_from_xl_files_color()
class UnifiedRecord:
    def __init__(self):
       self.weight_loss = None
       self.co2 = None
       self.o2 = None
       self.TSS = None
       self.texture_firmness = None
       self.color = None
       self.bacteria_total = None
       self.yeasts = None
       self.lactic = None
#the idea here is that instead of haveing 5 different variables we combine all of the sets into one big set
#its possible as all sets share keys (date(day),type of bacteria used)
#after that method we only need to use this single variable for all data
#it will also allow us to reset the data after we predict the future incase we want too
def build_unified_data(d_weight, d_texture, d_tss, d_o2, d_co2, d_morph, d_bac_y, d_bac_l, d_color):
    unified_data = {}

    all_keys = (set(d_weight.keys()) | set(d_texture.keys()) | set(d_tss.keys()) |
                set(d_o2.keys()) | set(d_co2.keys()) | set(d_morph.keys()) |
                set(d_bac_y.keys()) | set(d_bac_l.keys()) | set(d_color.keys()))

    for key in all_keys:
        rec = UnifiedRecord()


        if key in d_weight:
            val = d_weight[key]
            rec.weight_loss = val.weight_loss if hasattr(val, 'weight_loss') else val
        if key in d_co2:
            val = d_co2[key]
            rec.co2 = val.co2 if hasattr(val, 'co2') else val
        if key in d_o2:
            val = d_o2[key]
            rec.o2 = val.o2 if hasattr(val, 'o2') else val
        if key in d_tss:
            val = d_tss[key]
            rec.TSS = val.TSS if hasattr(val, 'TSS') else val
        if key in d_texture:
            val = d_texture[key]
            rec.texture_firmness = val.force if hasattr(val, 'force') else val
        if key in d_morph:
            val = d_morph[key]
            rec.morphology_rate = val.morphology_rate if hasattr(val, 'morphology_rate') else val
        if key in d_color:
            val = d_color[key]
            rec.color = val.color if hasattr(val, 'color') else val


        if key in d_bac_y:
            val = d_bac_y[key]
            if hasattr(val, 'bacteria_total'):
                rec.bacteria_total = val.bacteria_total
                rec.yeasts = val.yeasts
            else:
                rec.bacteria_total = val[0]
                rec.yeasts = val[1]

        if key in d_bac_l:
            val = d_bac_l[key]
            if hasattr(val, 'lactic'):
                rec.bacteria_total = val.bacteria_total
                rec.lactic = val.lactic
            else:
                rec.bacteria_total = val[0]
                rec.lactic = val[1]

        unified_data[key] = rec

    return unified_data

current_hash_data = build_unified_data(dict_weight, dict_texture, dict_tss, dict_o2, dict_co2, dict_morphology, dict_bac_yeast, dict_bac_lactic, dict_color)


In [17]:
def get_max_real_day(data_map):
    if not data_map: return 0
    return int(max(day for day, treatment in data_map.keys()))

MAX_REAL_DAY = get_max_real_day(current_hash_data)

In [18]:
  W_WEIGHT_LOSS = 1.0
  W_CO2 = 1.0
  W_O2 = 1.0
  W_TEXTURE = 1.0
  W_COLOR = 1.0
  W_MORPHOLOGY = 1.0
  W_BACTERIA = 1.0

In [19]:


def get_avg_val(val):
    if val is None: return None
    if isinstance(val, list):
        valid = [v for v in val if pd.notna(v)]
        return sum(valid) / len(valid) if valid else None
    return val
#def get_avg_val(val):
def calculate_freshness_score(day, treatment, obj, full_data):

    NUM_METRICS = 7  # (Weight loss, CO2, O2, Texture, Color, Morphology, Bacteria total)


    MAX_PENALTY = 100.0 / NUM_METRICS

    day0_obj = full_data.get((0, treatment))
    co2_day0 = get_avg_val(getattr(day0_obj, 'co2', None)) if day0_obj else None
    o2_day0 = get_avg_val(getattr(day0_obj, 'o2', None)) if day0_obj else None
    color_day0 = get_avg_val(getattr(day0_obj, 'color', None)) if day0_obj else None
    morphology_day0 = get_avg_val(getattr(day0_obj, 'morphology_rate', None)) if day0_obj else None
    bacteria_day0 = get_avg_val(getattr(day0_obj, 'bacteria_total', None)) if day0_obj else None

    weight_loss = get_avg_val(getattr(obj, 'weight_loss', None))
    co2 = get_avg_val(getattr(obj, 'co2', None))
    o2 = get_avg_val(getattr(obj, 'o2', None))
    texture = get_avg_val(getattr(obj, 'texture_firmness', None))
    color = get_avg_val(getattr(obj, 'color', None))
    morphology = get_avg_val(getattr(obj, 'morphology_rate', None))
    bacteria = get_avg_val(getattr(obj, 'bacteria_total', None))

    if all(v is None for v in [weight_loss, co2, o2, texture, color, morphology, bacteria]):
        return None

    score = 100.0


    if weight_loss is not None and weight_loss > 0:
        weight_deg_pct = weight_loss * 10
        score -= min(MAX_PENALTY, weight_deg_pct * (MAX_PENALTY / 100))


    if co2 is not None and co2_day0 and co2_day0 > 0:
        co2_change_pct = ((co2 - co2_day0) / co2_day0) * 100
        if co2_change_pct > 0:
            score -= min(MAX_PENALTY, co2_change_pct * (MAX_PENALTY / 100))


    if o2 is not None and o2_day0 and o2_day0 > 0:
        o2_loss_pct = ((o2_day0 - o2) / o2_day0) * 100
        if o2_loss_pct > 0:
            score -= min(MAX_PENALTY, o2_loss_pct * (MAX_PENALTY / 100))


    if texture is not None and day0_obj:
        texture_day0 = get_avg_val(getattr(day0_obj, 'texture_firmness', None))
        if texture_day0 and texture_day0 > 0:
            texture_loss_pct = ((texture_day0 - texture) / texture_day0) * 100
            if texture_loss_pct > 0:
                score -= min(MAX_PENALTY, texture_loss_pct * (MAX_PENALTY / 100))


    if color is not None and color_day0 and color_day0 > 0:
        color_change_pct = ((color - color_day0) / color_day0) * 100
        if color_change_pct > 0:
            score -= min(MAX_PENALTY, color_change_pct * (MAX_PENALTY / 100))


    if morphology is not None and morphology_day0 and morphology_day0 > 0:
        morph_change_pct = ((morphology - morphology_day0) / morphology_day0) * 100
        if morph_change_pct > 0:
            score -= min(MAX_PENALTY, morph_change_pct * (MAX_PENALTY / 100))


    if bacteria is not None and bacteria_day0 and bacteria_day0 > 0:
        bacteria_change_pct = ((bacteria - bacteria_day0) / bacteria_day0) * 100
        if bacteria_change_pct > 0:
            score -= min(MAX_PENALTY, bacteria_change_pct * (MAX_PENALTY / 100))

    return max(0.0, min(100.0, score))
#def calculate_freshness_score(obj):
def update_freshness_graph():
    data_list = []
    for (day, treatment), obj in current_hash_data.items():
        fresh_score= calculate_freshness_score(day, treatment, obj, current_hash_data)
        if fresh_score is not None:
            data_list.append({"day": day, "treatment": treatment, "value": fresh_score})

    df = pd.DataFrame(data_list)
    if df.empty: return px.line(title="No Data")


    df = df.sort_values(by=["treatment", "day"])

    df['value'] = df.groupby('treatment')['value'].cummin()
    control_scores = dict(zip(df[df['treatment'].str.lower() == 'control']['day'], df[df['treatment'].str.lower() == 'control']['value']))
    df['value'] = df.apply(lambda r: r['value'] - control_scores.get(r['day'], r['value']) if r['day'] in control_scores else 0.0, axis=1)

    df = df.sort_values(by="day")
    df['Data Type'] = df['day'].apply(lambda d: 'Real Data' if d <= MAX_REAL_DAY else 'Prediction')
    fig = px.line(df, x="day", y="value", color="treatment", line_dash="Data Type", markers=True,
                  title="Freshness to Time", line_dash_map={'Real Data': 'solid', 'Prediction': 'dash'})
    fig.update_layout(height=600, xaxis_title="Time (Days)", yaxis_title="Freshness Difference from Control")
    return fig
#def update_freshness_graph():
def generate_freshness_heatmap():
    data_list = []
    for (day, treatment), obj in current_hash_data.items():
        score = calculate_freshness_score(day, treatment, obj, current_hash_data)
        if score is not None:
            data_list.append({"day": int(day), "treatment": str(treatment), "score": score})

    df = pd.DataFrame(data_list)
    if df.empty: return px.imshow([[0]], title="No Data")
    #start showing graph only from the first day where we have all the data on
    valid_days_count = df.groupby('day')['treatment'].nunique()
    max_treatments = df['treatment'].nunique()
    full_days = valid_days_count[valid_days_count == max_treatments].index
    if not full_days.empty:
        first_valid_day = full_days.min()
        df = df[df['day'] >= first_valid_day]
    ######
    df = df.sort_values(by=["treatment", "day"])
    df['score'] = df.groupby('treatment')['score'].cummin()
    control_scores_hm = dict(zip(df[df['treatment'].str.lower() == 'control']['day'], df[df['treatment'].str.lower() == 'control']['score']))
    df['score'] = df.apply(lambda r: r['score'] - control_scores_hm.get(r['day'], r['score']) if r['day'] in control_scores_hm else 0.0, axis=1)

    pivot_df = df.pivot(index="treatment", columns="day", values="score")
    row_order = ["control", "CMC-3-methyl", "CMC-3-methyl+A", "CMC-3-methyl+B", "CMC-3-methyl+C"]
    existing_order = [t for t in row_order if t in pivot_df.index] + [t for t in pivot_df.index if t not in row_order]
    pivot_df = pivot_df.reindex(existing_order)
    fig = px.imshow(pivot_df,
                    labels=dict(x="Time (Days)", y="Treatment type", color="Freshness score"),
                    x=pivot_df.columns,
                    y=pivot_df.index,
                    color_continuous_scale="RdYlGn",

                    text_auto=".1f",
                    aspect="auto",
                    title="")

    fig.update_layout(height=600, xaxis=dict(tickmode='linear'))
    return fig

In [20]:
def apply_predictions_from_ui(target_day):
    global current_hash_data, dict_weight, dict_texture, dict_tss, dict_o2, dict_co2
    global dict_morphology, dict_bac_yeast, dict_bac_lactic, dict_color
    skip_value = 1#i dont remmember why i added this but i may remember later so ill keep it , i mean it allows to create a graph while skipping every X days, but should i implement that?

    new_weight = predict_Trendline_one_value_for_key(dict_weight, target_day, skip_value)
    new_texture = predict_Trendline_one_value_for_key(dict_texture, target_day, skip_value)
    new_tss = predict_Trendline_one_value_for_key(dict_tss, target_day, skip_value)
    new_o2 = predict_Trendline_one_value_for_key(dict_o2, target_day, skip_value)
    new_co2 = predict_Trendline_one_value_for_key(dict_co2, target_day, skip_value)
    new_morphology = predict_Trendline_one_value_for_key(dict_morphology, target_day, skip_value)
    new_color = predict_Trendline_one_value_for_key(dict_color, target_day, skip_value)
    new_bac_yeast = predict_Trendline_two_values_per_key(dict_bac_yeast, target_day, skip_value)
    new_bac_lactic = predict_Trendline_two_values_per_key(dict_bac_lactic, target_day, skip_value)
    # now we rebuild the dictionary with updated predicted values, later if we will want the original directory we can just build it again from original values
    current_hash_data = build_unified_data(new_weight, new_texture, new_tss, new_o2, new_co2, new_morphology, new_bac_yeast, new_bac_lactic, new_color)
    msg = f"pridicted until: {int(target_day)}."
    preview_fig = get_fig_from_map("o2", current_hash_data)
    return msg, preview_fig


In [21]:
#this method allows to delete prediction in order to reset the graphs
def reset_predictions_from_ui():
    global current_hash_data, dict_weight, dict_texture, dict_tss, dict_o2, dict_co2
    global dict_morphology, dict_bac_yeast, dict_bac_lactic, dict_color


    dict_weight = collect_data_from_xl_files_weight()
    dict_texture = collect_data_from_xl_files_texture_analyzer()
    dict_tss = collect_data_from_xl_files_tss()
    dict_o2 = collect_data_from_xl_files_o2()
    dict_co2 = collect_data_from_xl_files_co2()
    dict_morphology = collect_data_from_xl_files_morphology()
    dict_bac_yeast = collect_data_from_xl_files_bacteria_total_yeasts()
    dict_bac_lactic = collect_data_From_Xl_files_bacteria_total_lactic()
    dict_color = collect_data_from_xl_files_color()


    current_hash_data = build_unified_data(dict_weight, dict_texture, dict_tss, dict_o2, dict_co2, dict_morphology, dict_bac_yeast, dict_bac_lactic, dict_color)

    msg = "Predictions deleted."
    preview_fig = get_fig_from_map("o2", current_hash_data)
    return msg, preview_fig

In [22]:
def auto_advance(step):
    if not global_spatial_frames: return 0
    next_step = (int(step) + 1) % len(global_spatial_frames)
    return next_step

In [23]:
# #זה מודל שעובד על הרעיון שהחיידקים הלקטים אוכלים את החסה ומתרבים לאט לאט, הבנתי מלילך שזה אינו נכון אבל למקרה ויהיה מה להוסיף מכאן בחזרה אני שומר את הקוד

# class MicrobialCompetitionABM:
#     def __init__(self, width=40, height=40, yeast_count=20, lactic_count=20):
#         self.width = width
#         self.height = height
#         self.grid = np.zeros((height, width), dtype=int)
#         self.health_grid = np.full((height, width), 100.0)
#         self.acid_grid = np.zeros((height, width))
#         self.entities = {}
#         for _ in range(int(yeast_count)): self._spawn('yeast', 1)
#         for _ in range(int(lactic_count)): self._spawn('lactic', 2)

#         self.history = {'grid': [self.grid.copy()], 'acid': [self.acid_grid.copy()], 'yeast': [yeast_count], 'lactic': [lactic_count]}

#     def _spawn(self, etype, grid_val):
#         while True:
#             x = random.randint(0, self.width - 1)
#             y = random.randint(0, self.height - 1)
#             if self.grid[y, x] == 0:
#                 self.entities[len(self.entities)] = {'type': etype, 'x': x, 'y': y, 'energy': 10, 'age': 0}
#                 self.grid[y, x] = grid_val
#                 break

#     def _get_neighbors(self, x, y, radius=1):
#         neighbors = []
#         for dx in range(-radius, radius + 1):
#             for dy in range(-radius, radius + 1):
#                 if dx == 0 and dy == 0: continue
#                 nx, ny = (x + dx) % self.width, (y + dy) % self.height
#                 neighbors.append((nx, ny))
#         return neighbors

#     def _get_empty_healthy(self, x, y):
#         return [(nx, ny) for nx, ny in self._get_neighbors(x, y, 1) if self.grid[ny, nx] == 0 and self.health_grid[ny, nx] > 0]

#     def step(self):
#         new_ents = {}
#         to_remove = set()

#         for eid, ent in self.entities.items():
#             if eid in to_remove: continue
#             x, y = ent['x'], ent['y']

#             # אם אין משאבים התא מת
#             if self.health_grid[y, x] <= 0:
#                 to_remove.add(eid)
#                 continue

#             ent['age'] += 1
#             if ent['age'] >= 80:
#                 to_remove.add(eid)
#                 self.grid[y, x] = 0
#                 continue

#             empty = self._get_empty_healthy(x, y)

#             if ent['type'] == 'yeast':
#                 self.health_grid[y, x] -= 8.0
#                 bonus = 2.0 if self.health_grid[y, x] < 60 else 1.0
#                 ent['energy'] += 3.0 * bonus
#                 if self.acid_grid[y, x] > 5:
#                     ent['energy'] -= 5.0
#                 self.health_grid[y, x] -= 8.0
#                 in_acid = any(self.grid[ny, nx] == 2 for nx, ny in self._get_neighbors(x, y, 2))

#                 if empty:
#                     self.grid[y, x] = 0
#                     nx, ny = random.choice(empty)
#                     ent['x'], ent['y'] = nx, ny
#                     self.grid[ny, nx] = 1
#                     ent['energy'] -= 0.5

#                     if ent['energy'] >= 15 and (not in_acid or random.random() > 0.5):
#                         ent['energy'] //= 2
#                         spawn = self._get_empty_healthy(nx, ny)
#                         if spawn:
#                             cx, cy = random.choice(spawn)
#                             new_ents[max(list(self.entities.keys()) + list(new_ents.keys()) + [0]) + 1] = {'type': 'yeast', 'x': cx, 'y': cy, 'energy': ent['energy'], 'age': 0}
#                             self.grid[cy, cx] = 1

#             elif ent['type'] == 'lactic':
#                 ent['energy'] += 2.0
#                 for nx, ny in self._get_neighbors(x, y, 1):
#                     self.acid_grid[ny, nx] += 2.0
#                 if empty:
#                     self.grid[y, x] = 0
#                     nx, ny = random.choice(empty)
#                     ent['x'], ent['y'] = nx, ny
#                     self.grid[ny, nx] = 2
#                     ent['energy'] -= 0.5

#                     if ent['energy'] >= 15:
#                         ent['energy'] //= 2
#                         spawn = self._get_empty_healthy(nx, ny)
#                         if spawn:
#                             cx, cy = random.choice(spawn)
#                             new_ents[max(list(self.entities.keys()) + list(new_ents.keys()) + [0]) + 1] = {'type': 'lactic', 'x': cx, 'y': cy, 'energy': ent['energy'], 'age': 0}
#                             self.grid[cy, cx] = 2

#             if ent['energy'] <= 0:
#                 to_remove.add(eid)
#                 self.grid[ent['y'], ent['x']] = 0

#         for eid in to_remove:
#             if eid in self.entities: del self.entities[eid]
#         self.entities.update(new_ents)

#         self.grid[self.health_grid <= 0] = 3


#         self.history['grid'].append(self.grid.copy())
#         self.history['yeast'].append(sum(1 for e in self.entities.values() if e['type'] == 'yeast'))
#         self.history['lactic'].append(sum(1 for e in self.entities.values() if e['type'] == 'lactic'))
#         self.history['acid'].append(self.acid_grid.copy())
# global_spatial_history = []
# global_pop_history = {'yeast': [], 'lactic': []}
# global_acid_history = []
# global_spatial_frames = []
# def run_and_save_spatial(yeast_init, lactic_init, steps):
#     global global_pop_history
#     model = MicrobialCompetitionABM(width=40, height=40, yeast_count=int(yeast_init), lactic_count=int(lactic_init))

#     output_video_path = "./spatial_simulation.mp4"
#     frames = []

#     cmap = ListedColormap(['#f8f8f8', '#ef4444', '#3b82f6', '#1f2937'])


#     for current_step in range(int(steps)):
#         model.step()

#         fig, ax = plt.subplots(figsize=(6, 6))
#         ax.imshow(model.grid, cmap=cmap, vmin=0, vmax=3)

#         if np.max(model.acid_grid) > 0:
#             ax.imshow(model.acid_grid, cmap='YlOrBr', alpha=0.3, vmin=0, vmax=10)

#         ax.axis('off')


#         fig.canvas.draw()
#         rgba_buffer = fig.canvas.buffer_rgba()
#         rgb_image = np.asarray(rgba_buffer)[:, :, :3]

#         frames.append(rgb_image)
#         plt.close(fig)


#     imageio.mimsave(output_video_path, frames, fps=5, macro_block_size=None)

#     global_pop_history['yeast'] = model.history['yeast']
#     global_pop_history['lactic'] = model.history['lactic']

#     df_pop = pd.DataFrame({
#         'Step': range(len(global_pop_history['yeast'])),
#         'Yeasts': global_pop_history['yeast'],
#         'Lactic': global_pop_history['lactic']
#     })
#     fig_line = px.line(df_pop, x='Step', y=['Yeasts', 'Lactic'],
#                        color_discrete_map={'Yeasts': '#ef4444', 'Lactic': '#3b82f6'},
#                        title="Population Dynamics")


#     return output_video_path, fig_line, gr.update(interactive=False)

# def render_spatial_frame(step):

#     return None


In [24]:
class MicrobialCompetitionABM:
    def __init__(self, width=40, height=40, yeast_count=20, lactic_count=20):
        self.width = width
        self.height = height
        self.grid = np.zeros((height, width), dtype=int)
        self.health_grid = np.full((height, width), 100.0)
        self.acid_grid = np.zeros((height, width))
        self.entities = {}


        for _ in range(int(yeast_count)):
            self._spawn('yeast', 1, init_energy=10)


        for _ in range(int(lactic_count)):
            self._spawn('lactic', 2, init_energy=30)

        self.history = {'grid': [self.grid.copy()], 'acid': [self.acid_grid.copy()], 'yeast': [yeast_count], 'lactic': [lactic_count]}

    def _spawn(self, etype, grid_val, init_energy):
        while True:
            x = random.randint(0, self.width - 1)
            y = random.randint(0, self.height - 1)
            if self.grid[y, x] == 0:

                self.entities[len(self.entities)] = {'type': etype, 'x': x, 'y': y, 'energy': init_energy, 'age': 0}
                self.grid[y, x] = grid_val
                break

    def _get_neighbors(self, x, y, radius=1):
        neighbors = []
        for dx in range(-radius, radius + 1):
            for dy in range(-radius, radius + 1):
                if dx == 0 and dy == 0: continue
                nx, ny = (x + dx) % self.width, (y + dy) % self.height
                neighbors.append((nx, ny))
        return neighbors

    def _get_empty_healthy(self, x, y):
        return [(nx, ny) for nx, ny in self._get_neighbors(x, y, 1) if self.grid[ny, nx] == 0 and self.health_grid[ny, nx] > 0]


    def step(self):
        new_ents = {}
        to_remove = set()
        self.acid_grid = self.acid_grid * 0.93
        self.acid_grid[self.acid_grid < 0.1] = 0.0

        for eid, ent in self.entities.items():
            if eid in to_remove: continue
            x, y = ent['x'], ent['y']

            if self.health_grid[y, x] <= 0 and ent['type'] == 'yeast':
                to_remove.add(eid)
                self.grid[y, x] = 0
                continue

            ent['age'] += 1


            if ent['type'] == 'yeast':
                if ent['age'] >= 60:
                    if random.random() < 0.10:
                        to_remove.add(eid)
                        self.grid[y, x] = 0
                        continue


            if ent['type'] == 'lactic':
                if ent['age'] < 20:
                    lactic_death_chance = 0.01
                elif ent['age'] < 40:
                    lactic_death_chance = 0.04
                else:
                    lactic_death_chance = 0.10


                if self.health_grid[y, x] < 40:
                    lactic_death_chance += 0.08


                if random.random() < lactic_death_chance:
                    to_remove.add(eid)
                    self.grid[y, x] = 0
                    continue

            empty = self._get_empty_healthy(x, y)


            if ent['type'] == 'yeast':
                self.health_grid[y, x] -= 6.5
                bonus = 1.5 if self.health_grid[y, x] < 50 else 1.0
                ent['energy'] += 1.5 * bonus

                if self.acid_grid[y, x] > 3:
                    ent['energy'] -= 6.0

                if empty:
                    self.grid[y, x] = 0
                    nx, ny = random.choice(empty)
                    ent['x'], ent['y'] = nx, ny
                    self.grid[ny, nx] = 1
                    ent['energy'] -= 0.5

                    if ent['energy'] >= 22:
                        ent['energy'] //= 2
                        spawn = self._get_empty_healthy(nx, ny)
                        if spawn:
                            cx, cy = random.choice(spawn)
                            new_ents[max(list(self.entities.keys()) + list(new_ents.keys()) + [0]) + 1] = {'type': 'yeast', 'x': cx, 'y': cy, 'energy': ent['energy'], 'age': 0}
                            self.grid[cy, cx] = 1


            elif ent['type'] == 'lactic':
                for nx, ny in self._get_neighbors(x, y, 1):
                    self.acid_grid[ny, nx] += 1.0

                decay_rate = 0.8
                if self.health_grid[y, x] > 50:
                    decay_rate -= 0.3

                ent['energy'] -= decay_rate

                if empty and ent['energy'] > 0:
                    self.grid[y, x] = 0
                    nx, ny = random.choice(empty)
                    ent['x'], ent['y'] = nx, ny
                    self.grid[ny, nx] = 2

            if ent['energy'] <= 0:
                to_remove.add(eid)
                self.grid[ent['y'], ent['x']] = 0

        for eid in to_remove:
            if eid in self.entities: del self.entities[eid]
        self.entities.update(new_ents)

        self.grid[self.health_grid <= 0] = 3

        self.history['grid'].append(self.grid.copy())
        self.history['yeast'].append(sum(1 for e in self.entities.values() if e['type'] == 'yeast'))
        self.history['lactic'].append(sum(1 for e in self.entities.values() if e['type'] == 'lactic'))
        self.history['acid'].append(self.acid_grid.copy())

global_pop_history = {'yeast': [], 'lactic': []}
def run_and_save_spatial(yeast_init, lactic_init, steps):
    global global_pop_history
    model = MicrobialCompetitionABM(width=40, height=40, yeast_count=int(yeast_init), lactic_count=int(lactic_init))

    output_video_path = "./spatial_simulation.mp4"
    frames = []
    cmap = ListedColormap(['#f8f8f8', '#ef4444', '#3b82f6', '#1f2937'])

    for current_step in range(int(steps)):
        model.step()

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(model.grid, cmap=cmap, vmin=0, vmax=3)

        if np.max(model.acid_grid) > 0:
            ax.imshow(model.acid_grid, cmap='YlOrBr', alpha=0.3, vmin=0, vmax=10)

        ax.axis('off')

        fig.canvas.draw()
        rgba_buffer = fig.canvas.buffer_rgba()
        rgb_image = np.asarray(rgba_buffer)[:, :, :3]

        frames.append(rgb_image)
        plt.close(fig)

    imageio.mimsave(output_video_path, frames, fps=5, macro_block_size=None)

    global_pop_history['yeast'] = model.history['yeast']
    global_pop_history['lactic'] = model.history['lactic']

    df_pop = pd.DataFrame({
        'Step': range(len(global_pop_history['yeast'])),
        'Total Bacteria': global_pop_history['yeast'],
        'Lactic': global_pop_history['lactic']
    })
    fig_line = px.line(df_pop, x='Step', y=['Total Bacteria', 'Lactic'],
                       color_discrete_map={'Total Bacteria': '#ef4444', 'Lactic': '#3b82f6'},
                       title="Population Dynamics")
    fig_line.update_layout(
        xaxis=dict(range=[0, int(steps)]),
        yaxis_title="Population Size",
        legend_title_text="Microbe Type"
    )
    return output_video_path, fig_line